# Human fib/dec pseudotime — explicit root selection + Palantir

Analogous to the mouse root-selection notebook, adapted for human. Same idea: make the
root an **explicit, visible choice** (star it in diffusion space, don't scan a table),
then compute pseudotime — here with **Palantir**, run two ways for comparison.

**What's different for human**
- No fine subtypes: `cell_subtype` == `cell_type` (endometrial_fibroblast / decidual_cell),
  so "subtype rank" is just the fib-before-dec coarse separation. The meaningful
  orientation metric is **Spearman(pt, day)** across the 2 timepoints (secretory=0,
  menstrual=1).
- No `efib_proliferative` anchor exists, so that mouse-specific candidate is skipped
  (guarded). Root candidates that DO apply: global DC extreme, earliest-timepoint
  centroid, max-proliferation fibroblast, min-apoptosis fibroblast.
- The object already carries `X_diffmap`, `X_scVI`, and the neighbor graph, so DPT and
  Palantir-on-RTK-basis need no recompute; only Palantir-standard rebuilds a diffusion
  map from `X_scVI`.

**Palantir, two ways**
1. `palantir_pt_rtk` — feed Palantir the existing diffusion eigenbasis (X_diffmap, DC1
   dropped) as its multiscale space. Same geometry as DPT, different pseudotime algorithm.
2. `palantir_pt_std` — let Palantir build its own diffusion map from `X_scVI`. The
   standard Palantir workflow.
Palantir's `early_cell` is an obs **name**, so we convert the positional root index.

In [ ]:
import numpy as np, pandas as pd, scanpy as sc, anndata as ad
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

REPO = "/n/lab_storage/eddy_lab/users/nhilgert/xmens_final"
H5AD = f"{REPO}/notebooks/pseudotime_human/v1_all_cells/results/2026_08_28_human_fibdec_pseudotime.h5ad"
CLUSTER_KEY = "cell_subtype"   # human: same as cell_type (2 levels)

# human has no fine subtypes; the ordered biology is just fibroblast -> decidual
FIB = ["endometrial_fibroblast"]
DEC = ["decidual_cell"]
SUBTYPE_ORDER = FIB + DEC
RANK = {s: i for i, s in enumerate(SUBTYPE_ORDER)}

adata = sc.read_h5ad(H5AD)
print(adata.shape)
print("obsm:", list(adata.obsm))
print("has diffmap:", "X_diffmap" in adata.obsm, "| has neighbors:", "neighbors" in adata.uns)
print("current iroot:", adata.uns.get("iroot"))
print("cell_subtype levels:", adata.obs[CLUSTER_KEY].nunique(), "->", list(adata.obs[CLUSTER_KEY].unique()))

## Helper: recompute DPT from a given root and score it

In [ ]:
labels = adata.obs[CLUSTER_KEY].astype(str)
day = adata.obs["day"].astype(float).to_numpy()
subrank = labels.map(RANK).to_numpy(dtype=float)

def recompute_dpt(root_idx):
    "Set the root and recompute DPT on the existing diffmap. Returns pseudotime array."
    adata.uns["iroot"] = int(root_idx)
    sc.tl.dpt(adata)
    return adata.obs["dpt_pseudotime"].to_numpy(float)

def score(pt, name, root_idx):
    ok = np.isfinite(pt) & np.isfinite(day)
    rho_day, p_day = spearmanr(pt[ok], day[ok])
    okr = np.isfinite(pt) & np.isfinite(subrank)
    rho_rank, _ = spearmanr(pt[okr], subrank[okr])
    med = (pd.DataFrame({"s": labels[np.isfinite(pt)], "pt": pt[np.isfinite(pt)]})
           .groupby("s")["pt"].median())
    med_ord = [med.get(s, np.nan) for s in SUBTYPE_ORDER]
    mono = np.all(np.diff([m for m in med_ord if np.isfinite(m)]) > 0)
    row = dict(candidate=name, root_idx=int(root_idx),
               root_subtype=labels.iloc[root_idx], root_day=float(day[root_idx]),
               rho_day=round(float(rho_day), 3), rho_subtype_rank=round(float(rho_rank), 3),
               fibdec_monotonic=bool(mono))
    return row, med

## Candidate roots - each is one explicit, named choice

Human-appropriate strategies. The `efib_proliferative` anchor is guarded and will skip.

In [ ]:
# DC2, DC3 = columns 1, 2 (column 0 is the trivial constant component).
dc = adata.obsm["X_diffmap"][:, [1, 2]]
dc2 = dc[:, 0]
candidates = {}

# --- 0. global DC2 extremes (geometric) ---
candidates["dc2_max (global)"] = int(dc2.argmax())
candidates["dc2_min (global)"] = int(dc2.argmin())

# --- 1. subtype anchor + diffmap extreme (mouse only — guarded) ---
anchor = "efib_proliferative"
amask = (labels == anchor).values
if amask.any():
    acand = np.where(amask)[0]
    centroid_all = dc[~amask].mean(axis=0)
    adist = np.linalg.norm(dc[amask] - centroid_all, axis=1)
    candidates["anchor_near"] = int(acand[adist.argmin()])
    candidates["anchor_far"]  = int(acand[adist.argmax()])
else:
    print(f"skip: anchor {anchor!r} absent (expected for human)")

# --- 2. earliest-timepoint centroid (secretory = day 0) ---
emask = (adata.obs["day"].astype(float) == day.min()).values
if emask.any():
    ecand = np.where(emask)[0]
    ecentroid = dc[emask].mean(axis=0)
    candidates[f"earliest_tp_centroid (day={day.min():g})"] = int(
        ecand[np.linalg.norm(dc[emask] - ecentroid, axis=1).argmin()])

# --- 3. max-proliferation fibroblast (cycling progenitor = start) ---
if "proliferation" in adata.obs.columns:
    fibmask = labels.isin(FIB).values
    prolif = adata.obs["proliferation"].to_numpy(float)
    pf = np.where(fibmask & np.isfinite(prolif))[0]
    if pf.size:
        candidates["max_prolif_fibroblast"] = int(pf[np.argmax(prolif[pf])])

# --- 4. min-apoptosis fibroblast (healthiest progenitor = start) ---
if "apoptosis" in adata.obs.columns:
    fibmask = labels.isin(FIB).values
    apop = adata.obs["apoptosis"].to_numpy(float)
    af = np.where(fibmask & np.isfinite(apop))[0]
    if af.size:
        candidates["min_apop_fibroblast"] = int(af[np.argmin(apop[af])])

# --- 5. current object root, for reference ---
candidates["current_object_root"] = int(adata.uns.get("iroot", 0))

for k, v in candidates.items():
    print(f"{k:30s} idx={v:6d}  subtype={labels.iloc[v]:22s}  tp={adata.obs['timepoint'].iloc[v]}")

## Look at the root in diffusion space (don't scan a table)

In [ ]:
# ---- choose which candidate to eyeball ----
choice = "dc2_max (global)"
root_idx = candidates[choice]

fig, ax = plt.subplots(figsize=(6.5, 6))
for lab in sorted(labels.unique()):
    m = (labels == lab).values
    ax.scatter(dc[m, 0], dc[m, 1], s=6, alpha=0.5, label=lab)
ax.scatter(dc[root_idx, 0], dc[root_idx, 1], s=260, c="red", marker="*",
           edgecolor="black", linewidth=1.3, zorder=5, label="root")
ax.set_xlabel("DC2"); ax.set_ylabel("DC3"); ax.set_title(f"root: {choice}")
ax.legend(markerscale=2, fontsize=8, loc="best")
fig.tight_layout(); plt.show()

print(f"chosen root: {choice}")
for k in ("obs_name","timepoint","day","cell_type","cell_subtype"):
    val = adata.obs_names[root_idx] if k=="obs_name" else adata.obs[k].iloc[root_idx]
    print(f"  {k:10s}= {val}")
print(f"  DC2, DC3  = {dc[root_idx]}")

### Root across several DC pairs

In [ ]:
from matplotlib.lines import Line2D
X = adata.obsm["X_diffmap"]
pairs = [(1, 2), (1, 3), (2, 3), (1, 4)]
n_comps = X.shape[1]
pairs = [(a, b) for (a, b) in pairs if a < n_comps and b < n_comps]
def dc_name(col): return f"DC{col + 1}"
ncol = 2; nrow = int(np.ceil(len(pairs) / ncol))

cats = sorted(labels.unique())
cmap = plt.get_cmap("tab10")
color_of = {c: cmap(i % cmap.N) for i, c in enumerate(cats)}
pt_colors = labels.map(color_of).to_numpy()

fig, axes = plt.subplots(nrow, ncol, figsize=(5.5 * ncol, 5 * nrow), squeeze=False)
for ax, (cx, cy) in zip(axes.ravel(), pairs):
    ax.scatter(X[:, cx], X[:, cy], c=pt_colors, s=6, alpha=0.5)
    ax.scatter(X[root_idx, cx], X[root_idx, cy], s=240, c="red", marker="*",
               edgecolor="black", linewidth=1.2, zorder=5)
    ax.set_xlabel(dc_name(cx)); ax.set_ylabel(dc_name(cy))
for ax in axes.ravel()[len(pairs):]:
    ax.axis("off")
handles = [Line2D([0],[0], marker="o", linestyle="", markersize=7,
                  markerfacecolor=color_of[c], markeredgecolor="none", label=c) for c in cats]
handles.append(Line2D([0],[0], marker="*", linestyle="", markersize=13,
                      markerfacecolor="red", markeredgecolor="black", label="root"))
fig.legend(handles=handles, loc="center right", fontsize=8)
fig.suptitle("Diffusion components — colored by cell subtype", y=1.01)
fig.tight_layout(rect=[0, 0, 0.86, 1]); plt.show()

## Rank all candidates by DPT (cheap sanity pass before Palantir)

In [ ]:
rows, meds = [], {}
for name, idx in candidates.items():
    pt = recompute_dpt(idx)
    row, med = score(pt, name, idx)
    rows.append(row); meds[name] = med
summary = pd.DataFrame(rows).set_index("candidate").sort_values("rho_day", ascending=False)
print("Ranked by Spearman(pt, day) — the meaningful metric for 2-timepoint human:\n")
print(summary.to_string())

## Commit the chosen root (writes the DPT column; Palantir cells add to it)

In [ ]:
CHOSEN = summary.index[0]           # <-- change to any key in `candidates`, or set CHOSEN_OBSNAME
CHOSEN_OBSNAME = None

if CHOSEN_OBSNAME is not None:
    root_idx = int(adata.obs_names.get_loc(CHOSEN_OBSNAME))
    root_desc = f"manual obs_name {CHOSEN_OBSNAME}"
else:
    root_idx = candidates[CHOSEN]
    root_desc = f"{CHOSEN} (idx {root_idx})"

pt = recompute_dpt(root_idx)
row, _ = score(pt, CHOSEN, root_idx)
adata.obs["dpt_pseudotime"] = pt
adata.obs["dpt_pseudotime_root"] = root_desc
print("committed root:", root_desc, "| subtype:", labels.iloc[root_idx], "| day:", day[root_idx])
print("  rho(pt,day) =", row["rho_day"], "| rho(pt,subtype_rank) =", row["rho_subtype_rank"])

## Palantir

`early_cell` is the obs **name** of the committed root. We run Palantir two ways and
orient each against real time.

In [ ]:
!uv pip install palantir

In [ ]:
import palantir
early_cell = adata.obs_names[root_idx]
print(f"early cell: {early_cell}  (subtype={labels.iloc[root_idx]}, tp={adata.obs['timepoint'].iloc[root_idx]})")

day_num = adata.obs["day"].astype(float).to_numpy()
def orient(pt, name):
    v = np.asarray(pt, float)
    ok = np.isfinite(v) & np.isfinite(day_num)
    rho, p = spearmanr(day_num[ok], v[ok])
    print(f"  {name}: Spearman(day, pt)={rho:+.4f} p={p:.2e}  "
          f"range [{np.nanmin(v):.3f}, {np.nanmax(v):.3f}]  {'OK' if rho > 0 else 'INVERTED'}")
    return rho

### Palantir mode 1 — on the existing diffusion eigenbasis (RTK/DPT geometry)

In [ ]:
ms_rtk = pd.DataFrame(
    adata.obsm["X_diffmap"][:, 1:],
    index=adata.obs_names,
    columns=[f"DC{i+1}" for i in range(1, adata.obsm["X_diffmap"].shape[1])],
)
pr_rtk = palantir.core.run_palantir(
    ms_rtk, early_cell=early_cell, num_waypoints=1200, seed=0,
    use_early_cell_as_start=True,
)
adata.obs["palantir_pt_rtk"] = pr_rtk.pseudotime.reindex(adata.obs_names).to_numpy()
adata.obs["palantir_entropy_rtk"] = pr_rtk.entropy.reindex(adata.obs_names).to_numpy()
orient(adata.obs["palantir_pt_rtk"], "RTK-basis")

### Palantir mode 2 — Palantir's own diffusion map from X_scVI

In [ ]:
dm = palantir.utils.run_diffusion_maps(
    pd.DataFrame(adata.obsm["X_scVI"], index=adata.obs_names),
    n_components=10, knn=30,
)
ms_std = palantir.utils.determine_multiscale_space(dm)
pr_std = palantir.core.run_palantir(
    ms_std, early_cell=early_cell, num_waypoints=1200, seed=0,
    use_early_cell_as_start=True,
)
adata.obs["palantir_pt_std"] = pr_std.pseudotime.reindex(adata.obs_names).to_numpy()
adata.obs["palantir_entropy_std"] = pr_std.entropy.reindex(adata.obs_names).to_numpy()
orient(adata.obs["palantir_pt_std"], "standard")

## Compare all pseudotimes — by day and by cell type

In [ ]:
import seaborn as sns
pt_cols = [c for c in ["dpt_pseudotime", "palantir_pt_rtk", "palantir_pt_std"] if c in adata.obs]
day_str = adata.obs["day"].astype(float).map(lambda d: f"{d:g}")
day_order = [f"{d:g}" for d in sorted(adata.obs["day"].astype(float).unique())]
celltype_col = "cell_subtype" if labels.nunique() > 2 else "cell_type"
ct = adata.obs[celltype_col].astype(str)

for col in pt_cols:
    pt = adata.obs[col]
    ok = np.isfinite(pt.to_numpy()) & np.isfinite(day_num)
    rho, p = spearmanr(day_num[ok], pt.to_numpy()[ok])
    tag = col.replace("palantir_pt_", "palantir-").replace("dpt_pseudotime", "DPT")
    print(f"[{tag}] Spearman(day, pt)={rho:+.4f}  p={p:.2e}  ({'OK' if rho>0 else 'INVERTED'})")
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    sns.violinplot(data=pd.DataFrame({"day": day_str.values, "pt": pt.values}),
                   x="day", y="pt", order=day_order, cut=0, ax=axes[0])
    axes[0].set_title(f"{tag}: by day (rho={rho:+.3f})"); axes[0].set_ylabel(f"pt ({tag})")
    ct_order = (pd.DataFrame({"ct": ct.values, "pt": pt.values})
                .groupby("ct")["pt"].median().sort_values().index.tolist())
    sns.violinplot(data=pd.DataFrame({"ct": ct.values, "pt": pt.values}),
                   x="ct", y="pt", order=ct_order, cut=0, ax=axes[1])
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha="right")
    axes[1].set_title(f"{tag}: by {celltype_col}")
    fig.tight_layout(); plt.show()

In [ ]:
# UMAP colored by each pseudotime
for col in pt_cols:
    sc.pl.embedding(adata, basis="X_umap", color=col, title=col)

## Correlation summary + write the rooted object

In [ ]:
ranks = labels.map(RANK).to_numpy(float)
print(f"{'column':22s}{'rho(day)':>12s}{'rho(subtype)':>14s}")
for col in pt_cols:
    pt = adata.obs[col].to_numpy(float)
    ok = np.isfinite(pt) & np.isfinite(ranks) & np.isfinite(day_num)
    rho_d, _ = spearmanr(pt[ok], day_num[ok])
    rho_r, _ = spearmanr(pt[ok], ranks[ok])
    print(f"{col:22s}{rho_d:>+12.4f}{rho_r:>+14.4f}")

out = f"{REPO}/notebooks/pseudotime_human/v1_all_cells/results/2026_08_28_human_fibdec_pseudotime_rooted.h5ad"
adata.write_h5ad(out)
print("\nwrote", out)